# Geldium Credit Delinquency — Preprocessing and EDA

This notebook loads `data/raw/dataset.csv`, performs data-quality checks, cleans categorical and numeric fields, engineers payment-history features, creates EDA charts, and saves `data/processed/processed.csv`. It does not modify the raw dataset or source Python files.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
warnings.filterwarnings('ignore')

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'dataset.csv'
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'processed.csv'
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Input:', RAW_PATH)
print('Output:', PROCESSED_PATH)

In [ ]:
if not RAW_PATH.exists():
    raise FileNotFoundError(f'Place dataset.csv at {RAW_PATH}')
df = pd.read_csv(RAW_PATH)
df.columns = df.columns.astype(str).str.strip().str.replace(' ', '_', regex=False)
print('Shape:', df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).to_frame('missing_count'))

In [ ]:
required = ['Delinquent_Account']
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise KeyError(f'Missing required columns: {missing_required}')

numeric_cols = ['Age', 'Income', 'Credit_Score', 'Credit_Utilization', 'Missed_Payments', 'Loan_Balance', 'Debt_to_Income_Ratio', 'Account_Tenure', 'Delinquent_Account']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df = df[df['Delinquent_Account'].isin([0, 1])].copy()
df['Delinquent_Account'] = df['Delinquent_Account'].astype(int)

def normalize_employment(value):
    if pd.isna(value):
        return 'unknown'
    value = str(value).strip().lower()
    if value in ['emp', 'employed', 'employee']:
        return 'employed'
    if 'self' in value:
        return 'self-employed'
    if 'unemploy' in value:
        return 'unemployed'
    if 'retir' in value:
        return 'retired'
    if 'student' in value:
        return 'student'
    return value

if 'Employment_Status' in df.columns:
    df['Employment_Status'] = df['Employment_Status'].apply(normalize_employment)
for col in ['Credit_Card_Type', 'Location']:
    if col in df.columns:
        df[col] = df[col].astype('string').str.strip().str.title()

In [ ]:
payment_cols = sorted([c for c in df.columns if c.startswith('Month_')], key=lambda c: int(c.split('_')[1]))
payment_map = {'on-time': 0, 'on time': 0, 'late': 1, 'missed': 2}
for col in payment_cols:
    df[col] = df[col].astype('string').str.strip().str.lower().map(payment_map)

if payment_cols:
    df['count_missed'] = (df[payment_cols] == 2).sum(axis=1)
    df['count_late'] = (df[payment_cols] == 1).sum(axis=1)
    df['count_on_time'] = (df[payment_cols] == 0).sum(axis=1)
    recent = payment_cols[-3:]
    df['recent_missed'] = (df[recent] == 2).sum(axis=1)
    df['recent_late'] = (df[recent] == 1).sum(axis=1)
    df['payment_risk_score'] = df[payment_cols].sum(axis=1)
else:
    for col in ['count_missed', 'count_late', 'count_on_time', 'recent_missed', 'recent_late', 'payment_risk_score']:
        df[col] = 0

for col in ['Income', 'Loan_Balance', 'Credit_Score']:
    if col in df.columns:
        df[f'{col.lower()}_missing'] = df[col].isna().astype(int)

display(df.head())
print('Processed shape:', df.shape)

In [ ]:
if 'Credit_Utilization' in df.columns:
    bins = [-np.inf, .30, .50, .75, 1.00, np.inf]
    labels = ['0–30%', '30–50%', '50–75%', '75–100%', 'Above 100%']
    df['Credit_Utilization_Band'] = pd.cut(df['Credit_Utilization'], bins=bins, labels=labels)
    chart = df.groupby('Credit_Utilization_Band', observed=False)['Delinquent_Account'].mean().mul(100)
    chart.plot(kind='bar', figsize=(8, 4), color='mediumpurple', title='Delinquency Rate by Credit Utilisation Band')
    plt.ylabel('Delinquency rate (%)')
    plt.tight_layout()
    plt.show()

if 'count_missed' in df.columns:
    chart = df.groupby('count_missed')['Delinquent_Account'].mean().mul(100)
    chart.plot(kind='bar', figsize=(8, 4), color='crimson', title='Delinquency Rate by Missed Payments in History')
    plt.ylabel('Delinquency rate (%)')
    plt.tight_layout()
    plt.show()

In [ ]:
df.to_csv(PROCESSED_PATH, index=False)
print(f'Saved processed data to: {PROCESSED_PATH}')
print('Final missing values:')
display(df.isna().sum().sort_values(ascending=False).to_frame('missing_count').head(15))